# ARK-019 V3 — R2-informed Hierarchical Capability Guardian

Select **T4 GPU**, then **Runtime → Run all**. This consumes the completed ARK-018 prepared cache and SCIENCE_ONLY checkpoints already in Drive. It fails closed before continuation arms if identity, exact-resume, or runtime gates fail.


In [ ]:
# CELL 0 — clone frozen executable / static tests / Drive substrate checks
import os, sys, json, subprocess
from pathlib import Path
REPO=Path('/content/An-Ra-the-new-AGI-r3')
REMOTE='https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git'
BRANCH='Arkenstone'
EXEC='63dc47d9bdcd85e004c03aaf9a738c9682fcd8f0'
if not REPO.exists(): subprocess.run(['git','clone','--branch',BRANCH,'--single-branch','--depth','80',REMOTE,str(REPO)],check=True)
else:
    subprocess.run(['git','-C',str(REPO),'fetch','origin',BRANCH,'--depth','80'],check=True)
subprocess.run(['git','-C',str(REPO),'checkout','-q',EXEC],check=True)
head=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip(); assert head==EXEC
expected={'experiments/ARK-019/ark019_v3_core.py':'bb815ebf1c6c32c45c211a744e0a4d19099dd542','experiments/ARK-019/run_ark019_v3.py':'ef41096069714668b3b1e5c0e165904908ebc885','tests/test_ark019_v3.py':'c52d6857aa2933c28291d4d115205af927c37c5f'}
for p,sha in expected.items():
    got=subprocess.check_output(['git','-C',str(REPO),'hash-object',p],text=True).strip(); assert got==sha,(p,got,sha)
subprocess.run([sys.executable,'-m','pip','install','-q','tokenizers==0.21.4','pytest','numpy'],check=True)
subprocess.run([sys.executable,'-m','py_compile',str(REPO/'experiments/ARK-019/ark019_v3_core.py'),str(REPO/'experiments/ARK-019/run_ark019_v3.py'),str(REPO/'experiments/ARK-018/ark018_v3_common.py'),str(REPO/'experiments/ARK-018/ark018_v3_binding_fast.py')],check=True)
subprocess.run([sys.executable,'-m','pytest',str(REPO/'tests/test_ark019_v3.py'),'-q'],cwd=REPO,check=True)
import torch
if not torch.cuda.is_available(): raise RuntimeError('Select Runtime -> Change runtime type -> T4 GPU')
print('GPU:',torch.cuda.get_device_name(0),'VRAM GiB:',round(torch.cuda.get_device_properties(0).total_memory/2**30,2))
sys.path.insert(0,str(REPO/'experiments/ARK-019')); sys.path.insert(0,str(REPO/'experiments/ARK-018'))
import run_ark019_v3 as R3
print('R3 import PASS | horizon',R3.C.HORIZON,'| arms',R3.C.ARMS)
from google.colab import drive
drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/genisis-arkenstone/ARK018_SCIENCE_BIRTH_V1')
required=[ROOT/'prepared/ARK-018_PREPARED_RECEIPT.json',ROOT/'prepared/tokenizer.json',ROOT/'prepared/train.bin',ROOT/'prepared/control.bin',ROOT/'prepared/sealed.bin',ROOT/'prepared/token_counts.npy',ROOT/'checkpoints/seed_31801/SCIENCE_ONLY.pt',ROOT/'checkpoints/seed_31902/SCIENCE_ONLY.pt']
missing=[str(p) for p in required if not p.exists()]
if missing: raise FileNotFoundError('Missing ARK-018 prerequisite(s): '+repr(missing))
print('ARK-019 V3 STATIC + SUBSTRATE PREFLIGHT: PASS')


In [ ]:
# CELL 1 — full prospective R3 campaign. Internal order: parents -> exact-resume smoke -> runtime gate -> 4 matched sets x 4 arms.
import subprocess, sys
cmd=[sys.executable,'experiments/ARK-019/run_ark019_v3.py','--mode','all']
print('Starting:', ' '.join(cmd), flush=True)
subprocess.run(cmd,cwd=REPO,check=True)


In [ ]:
# CELL 2 — verify final artifact / show decision / download
import hashlib,json
from pathlib import Path
from google.colab import files
OUT=Path('/content/drive/MyDrive/genisis-arkenstone/ARK019_GUARDIAN_V3')
bundle=OUT/'ARKENSTONE_ARK019_V3_GUARDIAN_RESULTS.zip'
if not bundle.exists(): raise FileNotFoundError(bundle)
actual=hashlib.sha256(bundle.read_bytes()).hexdigest()
side=OUT/(bundle.name+'.sha256')
if side.exists(): assert side.read_text().split()[0]==actual
result_path=OUT/'ARK-019_V3_RESULT.json'
if result_path.exists():
    r=json.loads(result_path.read_text()); print('STATUS:',r.get('status')); print('VERDICT:',r.get('decision',{}).get('verdict')); print('FLAGS:',r.get('decision',{}).get('flags')); print(json.dumps(r.get('decision',{}).get('guardian_details',{}),indent=2))
else:
    f=OUT/'ARK-019_V3_FAILURE.json'; print('No final result. Failure:',f.read_text() if f.exists() else 'unknown')
print('ZIP SHA256:',actual)
files.download(str(bundle))
